<style>
/* ===== LIGHT CYAN MODEL BUILDING STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
    max-width: 1320px;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">🧠 مرحلة بناء النموذج (Model Building)</div>
    <div class="eda-text-box">
        في هذه المرحلة يتم تنفيذ خطوة
        <span class="eda-highlight">بناء نموذج التعلم العميق</span>،
        حيث ننتقل من تحليل البيانات إلى إنشاء نموذج قادر على
        تعلم الأنماط واستخلاص الخصائص من الصور.
    </div>
    <div class="eda-text-box">
        يعتمد هذا النموذج على
        <span class="eda-highlight">تقنية التعلم بالنقل (Transfer Learning)</span>،
        من خلال استخدام نموذج مدرّب مسبقًا على مجموعة بيانات
        كبيرة، مما يساعد على تحسين الأداء وتقليل زمن التدريب.
    </div>
    <div class="eda-text-box">
        تتضمن هذه الخلية:
        <ul style="margin-right: 20px;">
            <li>تحميل النموذج الأساسي المدرب مسبقًا</li>
            <li>تجميد أوزان النموذج للحفاظ على الخصائص المستخرجة</li>
            <li>إضافة طبقات مخصصة للتصنيف</li>
            <li>إعداد النموذج لعملية التدريب</li>
            <li>تجهيز آلية لحفظ أفضل نموذج</li>
        </ul>
    </div>

</div>

<div class="eda-note">
    ℹ️ هذه الخلية تمثل الأساس الفعلي لعملية التعلم، حيث يتم فيها إنشاء النموذج الذي سيتم تدريبه وتقييمه في المراحل اللاحقة.
</div>

</div>


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint

# بناء النموذج باستخدام Transfer Learning
base_model = EfficientNetB3(
    weights='imagenet',      # استخدام أوزان ImageNet
    include_top=False,       # إزالة الطبقة العلوية
    input_shape=(224, 224, 3) # حجم الصورة المطلوب
)

# تجميد الطبقات الأساسية
base_model.trainable = False

# إضافة الطبقات العلوية
x = GlobalAveragePooling2D()(base_model.output)
output = Dense(11, activation='softmax')(x)  # 11 فئة لخلايا الدم

# إنشاء النموذج النهائي
model = Model(inputs=base_model.input, outputs=output)

# تجميع النموذج
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# إعداد مراقب لحفظ أفضل نموذج
checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

print("تم إنشاء النموذج بنجاح")
model.summary()

تم إنشاء النموذج بنجاح


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        960 │ block1a_se_excit

 Total params: 10,800,442 (41.20 MB)

 Trainable params: 16,907 (66.04 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

<style>
/* ===== LIGHT CYAN MODEL BUILDING STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
    max-width: 1320px;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">✅ فوائد هذه الخطوة</div>
    <div class="eda-text-box">
        <span class="eda-highlight">1. تسريع التدريب</span>
        <ul style="margin-right: 20px;">
            <li>استخدام المعرفة المسبقة المستفادة من بيانات ImageNet</li>
            <li>تقليل وقت التدريب بشكل كبير مقارنة بالتدريب من الصفر</li>
        </ul>
    </div>
    <div class="eda-text-box">
        <span class="eda-highlight">2. تحسين الأداء</span>
        <ul style="margin-right: 20px;">
            <li>استخراج ميزات بصرية متطورة وعالية الجودة</li>
            <li>الحصول على نتائج أفضل حتى عند توفر بيانات تدريب محدودة</li>
        </ul>
    </div>
    <div class="eda-text-box">
        <span class="eda-highlight">3. توفير الموارد</span>
        <ul style="margin-right: 20px;">
            <li>عدم الحاجة إلى تدريب نموذج تعلم عميق من البداية</li>
            <li>استهلاك أقل للذاكرة والطاقة الحاسوبية</li>
        </ul>
    </div>

</div>

<div class="eda-container">
    <div class="eda-title">📊 شرح أعمدة ملخص النموذج</div>
    <div class="eda-text-box">
        <span class="eda-highlight">1. Layer (الطبقة)</span>
        <ul style="margin-right: 20px;">
            <li>يمثل اسم كل طبقة مستخدمة داخل النموذج</li>
            <li>يساعد على فهم البنية التسلسلية للشبكة العصبية</li>
        </ul>
    </div>
    <div class="eda-text-box">
        <span class="eda-highlight">2. Output Shape (شكل الخرج)</span>
        <ul style="margin-right: 20px;">
            <li>يوضح أبعاد البيانات الناتجة بعد كل طبقة</li>
            <li>
                مثال:
                <span class="eda-highlight">(None, 224, 224, 3)</span>
                ويشير إلى صور بحجم 224×224 مع ثلاث قنوات لونية
            </li>
        </ul>
    </div>
    <div class="eda-text-box">
        <span class="eda-highlight">3. Param # (عدد المعاملات)</span>
        <ul style="margin-right: 20px;">
            <li>يمثل عدد الأوزان القابلة للتدريب داخل كل طبقة</li>
            <li>يعكس مدى تعقيد النموذج وحجمه الحسابي</li>
        </ul>
    </div>
    <div class="eda-text-box">
        <span class="eda-highlight">4. Connected to (متصل بـ)</span>
        <ul style="margin-right: 20px;">
            <li>يوضح كيفية ترابط الطبقات مع بعضها البعض</li>
            <li>يساعد في تتبع تدفق البيانات داخل النموذج</li>
        </ul>
    </div>

</div>

<div class="eda-note">
    ℹ️ فهم هذه الأعمدة ضروري لتحليل بنية النموذج، وتقييم تعقيده، وتشخيص أي مشاكل قد تظهر أثناء التدريب.
</div>

</div>


<style>
/* ===== LIGHT CYAN MODEL BUILDING STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
    max-width: 1320px;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">📊 تحليل معاملات النموذج (Model Parameters Analysis)</div>
    <div class="eda-text-box">
        يوضح ملخص النموذج أن العدد الكلي للمعاملات يبلغ
        <span class="eda-highlight">10,800,442 معامل</span>
        بحجم إجمالي قدره
        <span class="eda-highlight">41.20 ميجابايت</span>،
        وهو ما يشمل جميع الأوزان والقيم المستخدمة داخل النموذج.
    </div>
    <div class="eda-text-box">
        من هذا العدد، يوجد فقط
        <span class="eda-highlight">16,907 معامل قابلة للتدريب</span>
        بحجم
        <span class="eda-highlight">66.04 كيلوبايت</span>،
        وتمثل هذه القيم الأوزان الجديدة التي تمت إضافتها في هذه المرحلة،
        وتحديدًا طبقة
        <span class="eda-highlight">Dense</span>
        النهائية.
    </div>
    <div class="eda-text-box">
        في المقابل، يبلغ عدد المعاملات غير القابلة للتدريب
        <span class="eda-highlight">10,783,535 معامل</span>
        بحجم
        <span class="eda-highlight">41.14 ميجابايت</span>،
        وهي الأوزان الأصلية لنموذج
        <span class="eda-highlight">EfficientNetB3</span>
        التي تم تجميدها للحفاظ على المعرفة المستفادة مسبقًا.
    </div>
    <div class="eda-text-box">
        يعكس هذا التوزيع نهجًا فعالًا في
        <span class="eda-highlight">التعلم بالنقل</span>،
        حيث يتم الاحتفاظ بأكثر من
        <span class="eda-highlight">99.8%</span>
        من المعرفة المستفادة من بيانات ImageNet، مع تدريب جزء صغير فقط من النموذج،
        مما يؤدي إلى تقليل زمن التدريب وتحقيق أداء أفضل باستخدام موارد أقل.
    </div>

</div>

<div class="eda-note">
    ℹ️ هذا التحليل يوضح كفاءة تصميم النموذج وقدرته على تحقيق توازن مثالي بين الأداء العالي وسرعة التدريب.
</div>

</div>
